Let's start by discussing the difference between fermionic creation and annihilation operators and Majorana operators.
Starting from a set of fermionic operators $f_1^{(\dagger)},\cdots,f_N^{(\dagger)}$, for each index $j$ we build two (self-adjoint) Majorana operators 
\begin{equation}\gamma_j=f_j^\dagger+f_j, \quad \gamma_j^{\prime}=i\left(f_j^{\dagger}-f_j\right)\end{equation}
with commutation relations
$$\{ \gamma_i,\gamma_j\}= \{ \gamma_i^{\prime},\gamma_j^{\prime}\} = 2\delta_{ij},\qquad
    \{ \gamma_i^{\prime},\gamma_j\} =0.$$

Products of multiple Majorana operators are called Majorana strings, and we write them as 
$$\mu(v)\equiv(i)^{v^T \omega_{\mathrm{L}} v} \cdot \gamma_1^{v_1} {\gamma_1^\prime}^{v_2} \cdots \gamma_{N}^{v_{2N-1}}{\gamma_N^\prime}^{v_{2N}}$$
where 
$ v \in \{0,1\}^{2N}$ is the binary vector that uniquely determines $\mu(v)$ and $(i)^{v^T \omega_{\mathrm{L}} v}=1,i$ ensures that $\mu(v)$ remains hermitian for all possible $v$.

Majorana strings form a "nice basis" for fermionic operators because they satisy the following commutation relation
$$\mu(v) \mu\left(v^{\prime}\right)=(-1)^{v^T \omega v^{\prime}} \mu\left(v^{\prime}\right) \mu(v)$$
that allows us to write the effect of applying to $\mu(v)$ a rotation generated by $\mu(v')$ in a very analogous way to the case for qubits and the Pauli basis
$$ e^{i \frac{\theta}{2} \mu\left(v'\right)} \mu(v) e^{-i \frac{\theta}{2} \mu\left(v'\right)} =\left\{\begin{array}{lc}
\mu(v)\quad\quad\quad &\text { if } [\mu(v'), \mu(v)]=0 \\
\cos (\theta) \mu(v)+\sin (\theta)(-1)^{g\left(v, v'\right)} \mu\left(v+v'\right) &\text { if } \{\mu(v'), \mu(v)\}=0
\end{array}\right.$$

In a certain sense, Majorana strings are the equivalent in fermionic systems of Pauli strings for qubits.


Translating fermionic operators to (sums of) Majorana strings is done directly using the definition (1), so for example
$$n_i=f_i^\dagger f_i=\frac{1}{2}\left(1+i \gamma_i \gamma_i^{\prime}\right)\equiv\frac{1}{2}\left(1+\mu\left(v_{ii^\prime}\right)\right)$$
and
$$f_i^{\dagger} f_j+f_j^{\dagger} f_i=\frac{1}{2}\left(i\gamma_i \gamma_j^{\prime}-i\gamma_i^{\prime} \gamma_j\right)\equiv \frac12\left(\mu(v_{ij^\prime})-\mu(v_{i^\prime j})\right)$$
Both number operators and hopping terms are written in terms of 2 Majorana strings. More complicated operators split into more Majorana strings. For example density-density operators
$$n_i n_j = \frac14\left(1 + \mu(v_{ii^\prime}) +\mu(v_{jj^\prime}) - \mu(v_{ii^\prime jj^\prime})\right)$$
split into 4 strings.

Let's now take a look at how Majorana strings are implemented in the code

In [8]:
using Revise
using MajoranaPropagation
using PauliPropagation

Majorana strings are implemented by the `MajoranaString` type, what can be created by passing the non-zero indices of the binary vector $v$.

Let's start by creating $\mu\left(v_{11^\prime}\right)= i \gamma_1 \gamma_1{\prime}$

In [9]:
n_fermions = 6
mstring = MajoranaString(n_fermions, [1, 2])

110000000000

Strings are displayed from left ($v_1$) to right ($v_{2N}$). So $\mu\left(v_{22^\prime}\right)= i \gamma_2 \gamma_2{\prime}$ is encoded as `001100000000`

In [10]:
mstring = MajoranaString(n_fermions, [3, 4])

001100000000

Sums of Majorana strings are stored using the `MajoranaSum` type. Since most fermionic operators require multiple Majorana strings to represent them, this is the preferred way of manipulating such objects. For this type, some default constructors for the most common fermionic operators are implemented

In [11]:
n_fermions = 8

n3 = MajoranaSum(n_fermions, :n, 3)

MajoranaSum with 2 term(s):
    0.5 * 0000000000000000
    0.5 * 0000110000000000

In [12]:
hopping_1_and_4 = MajoranaSum(n_fermions, :hop, (1, 4))

MajoranaSum with 2 term(s):
    0.5 * 1000000100000000
    -0.5 * 0100001000000000

Building more complicated operators is as simple as

In [13]:
msum = n3 * hopping_1_and_4

MajoranaSum with 4 term(s):
    0.25 * 0100111000000000
    -0.25 * 1000110100000000
    0.25 * 1000000100000000
    -0.25 * 0100001000000000

Defining gate objects can be achieved in two ways
1. $\exp(-i \theta \mu(v)/2)$ for a single Majorana string $\mu(v)$ is represented as a `MajoranaRotation(mstring)` object 
2. a fermionic gate $\exp(-i \theta G/2)$ where $G$ is a fermionic operator (e.g. densities, hoppings, density-density operators) is done by calling `FermionicGate(symbol, indices)` 

Under the hood, `FermionicGate` is then translated into a sequence of `MajoranaRotation` with the corresponding strings generated by `symbol`.

In [14]:
n_fermions = 7
majorana_rotation = MajoranaRotation(MajoranaString(n_fermions, [5, 6])) # create the Majorana rotation corresponding to the non-trivial part of n_3
@show majorana_rotation

fermionic_gate = FermionicGate(:n, 3)
@show fermionic_gate

#we can check that they are equivalent
majorana_rotations, _ = MajoranaPropagation.getmajoranarotations(fermionic_gate, n_fermions)
@show majorana_rotations;

majorana_rotation = MajoranaRotation{UInt16}(00001100000000)
fermionic_gate = FermionicGate(:n, [3])
majorana_rotations = MajoranaRotation[MajoranaRotation{UInt16}(00001100000000)]


Finally, applying the gates to the `MajoranaSum` is done by calling `propagate!(gates, msum, rotation_angles)`

In [18]:
obs = MajoranaSum(n_fermions, :hop, [1 ,3])

propagate!([fermionic_gate], obs, [0.2])

MajoranaSum with 4 term(s):
    0.09933466539753061 * 01000100000000
    0.4900332889206208 * 10000100000000
    -0.4900332889206208 * 01001000000000
    0.09933466539753061 * 10001000000000